# K-IFRS 4-Step 검색 파이프라인 테스트

| Step | 설명 | 데이터 소스 |
|------|------|------------|
| 1 | 기준서 식별 | `standard_summaries.embedding` |
| 2 | Level 1 문단 검색 | `chunks` (authority=1) |
| 3 | IE 적용사례 | `paragraph_links` (source_component='ie') |
| 4 | BC 결론도출근거 | `paragraph_links` (source_component='bc') |

In [1]:
import psycopg
from pgvector.psycopg import register_vector
from ingester.embedder import Embedder

conn = psycopg.connect("dbname=kifrs", autocommit=True)
register_vector(conn)
embedder = Embedder()

print("DB 연결 OK")
print(f"Embedding model (passage): {embedder.model_passage}")
print(f"Embedding model (query):   {embedder.model_query}")

DB 연결 OK
Embedding model (passage): embedding-passage
Embedding model (query):   embedding-query


## Step 1: 기준서 식별
쿼리를 임베딩 → `standard_summaries` 테이블에서 코사인 유사도 top-5

In [2]:
def step1_identify_standard(query: str, top_k: int = 5):
    """Step 1: 쿼리에 가장 적합한 기준서 식별"""
    query_emb = embedder.embed_query(query)
    
    rows = conn.execute("""
        SELECT standard_id, title, 
               1 - (embedding <=> %s::vector) AS similarity
        FROM standard_summaries
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (query_emb, query_emb, top_k)).fetchall()
    
    print(f"Query: \"{query}\"\n")
    print(f"{'순위':<4} {'유사도':<8} {'기준서':<20} {'제목'}")
    print("-" * 70)
    for i, (sid, title, sim) in enumerate(rows, 1):
        print(f"{i:<4} {sim:.4f}   {sid:<20} {title}")
    
    return rows

# 테스트
results = step1_identify_standard("수행의무 판단 기준")

Query: "수행의무 판단 기준"

순위   유사도      기준서                  제목
----------------------------------------------------------------------
1    0.3203   K-IFRS 1115          고객과의 계약에서 생기는 수익
2    0.3038   K-IFRS 1029          초인플레이션 경제에서의 재무보고
3    0.2994   K-IFRS 1112          타 기업에 대한 지분의 공시
4    0.2969   실무서 2 중요성            실무서 2 중요성
5    0.2929   K-IFRS 1037          충당부채 우발부채 우발자산


## Step 2: Level 1 문단 검색
선택된 기준서 내에서 authority=1 청크를 벡터 검색. main → ag 순서로 그룹핑.

In [3]:
COMPONENT_ORDER = {"main": 0, "definitions": 1, "ag": 2, "transition": 3}

def step2_search_authoritative(query: str, standard_id: str, top_k: int = 10):
    """Step 2: 기준서 내 Level 1 문단 벡터 검색"""
    query_emb = embedder.embed_query(query)
    
    rows = conn.execute("""
        SELECT chunk_id, para_number, component, section_title,
               content_markdown,
               1 - (embedding <=> %s::vector) AS similarity
        FROM chunks
        WHERE standard_id = %s AND authority = 1
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (query_emb, standard_id, query_emb, top_k)).fetchall()
    
    # component 순서로 그룹핑
    rows_sorted = sorted(rows, key=lambda r: (COMPONENT_ORDER.get(r[2], 99), -r[5]))
    
    print(f"Query: \"{query}\" in {standard_id}\n")
    current_comp = None
    for chunk_id, para, comp, section, md, sim in rows_sorted:
        if comp != current_comp:
            comp_label = {"main": "본문", "ag": "적용지침", "definitions": "정의", "transition": "경과규정"}
            print(f"\n### {comp_label.get(comp, comp)} ({comp})")
            current_comp = comp
        preview = md[:150].replace('\n', ' ')
        print(f"  [{sim:.3f}] 문단 {para or 'N/A'} ({section or '-'})")
        print(f"         {preview}...")
    
    # 문단번호 목록 반환 (Steps 3-4에서 사용)
    para_numbers = [r[1] for r in rows if r[1]]
    return rows_sorted, para_numbers

# 테스트
chunks, para_nums = step2_search_authoritative("수행의무 판단 기준", "K-IFRS 1115")
print(f"\n검색된 문단번호: {para_nums}")

Query: "수행의무 판단 기준" in K-IFRS 1115


### 본문 (main)
  [0.393] 문단 32 (인식)
         32		문단 22~30에 따라 식별한 각 수행의무를 (문단 35~37에 따라) 기간에 걸쳐 이행하는지 또는 (문단 38에 따라) 한 시점에 이행하는지를 계약 개시시점에 판단한다. 수행의무가 기간에 걸쳐 이행되지 않는다면, 그 수행의무는 한 시점에 이행되는 것이다....
  [0.381] 문단 123 (공시)
         123		이 기준서를 적용하면서 내린, 고객과의 계약에서 생기는 수익의 금액과 시기의 결정에 유의적인 영향을 미친 판단과 그 판단의 변경을 공시한다. 특히 다음 모두를 결정할 때 내린 판단과 그 판단의 변경을 설명한다. 	⑴	수행의무의 이행 시기(문단 124~125 참...
  [0.381] 문단 B55 (공시)
         B55		라이선스가 구별되지 않는다면, 수행의무(약속한 라이선스 포함)가 기간에 걸쳐 이행되는 수행의무인지 한 시점에 이행되는 수행의무인지를 판단하기 위해 문단 31~38을 적용한다....
  [0.374] 문단 125 (공시)
         125		한 시점에 이행하는 수행의무에 대해, 약속한 재화나 용역을 고객이 언제 통제하는지를 검토하면서 내린 유의적인 판단을 공시한다. **거래가격과 수행의무에 배분하는 금액을 산정함**...
  [0.373] 문단 34 (인식)
         34		고객이 자산을 통제하는지를 판단할 때, 그 자산을 재매입하는 약정을 고려한다(문단 B64~B76 참조). **기간에 걸쳐 이행하는 수행의무**...
  [0.363] 문단 38 (인식)
         38		수행의무가 문단 35~37에 따라 기간에 걸쳐 이행되지 않는다면, 그 수행의무는 한 시점에 이행되는 것이다. 고객이 약속된 자산을 통제하고 기업이 수행의무를 이행하는 시점을 판단하기 위해, 문단 31~34의 통제에 관한 요구사항을 참고한다. 또 다음과 같은 통제...
  [0.3

## Step 3 & 4: IE 적용사례 / BC 결론도출근거
Step 2에서 찾은 문단번호를 `paragraph_links`에서 조회하여 관련 IE/BC 반환.

In [4]:
def step3_4_find_related(standard_id: str, para_numbers: list[str], component: str, top_k: int = 5):
    """Step 3/4: paragraph_links를 통해 관련 IE/BC 청크 조회"""
    if not para_numbers:
        print("문단번호 없음 — 건너뜀")
        return []
    
    # 문단번호 범위 매칭: target_para_start가 검색된 문단번호 중 하나와 일치
    placeholders = ",".join(["%s"] * len(para_numbers))
    rows = conn.execute(f"""
        SELECT DISTINCT c.chunk_id, c.para_number, c.section_title,
               c.content_markdown,
               pl.target_para_start, pl.target_para_end, pl.link_type
        FROM paragraph_links pl
        JOIN chunks c ON c.chunk_id = pl.source_chunk_id
        WHERE pl.standard_id = %s
          AND pl.source_component = %s
          AND pl.target_para_start IN ({placeholders})
        LIMIT %s
    """, (standard_id, component, *para_numbers, top_k)).fetchall()
    
    comp_label = "적용사례 (IE)" if component == "ie" else "결론도출근거 (BC)"
    print(f"\n## {comp_label} — {standard_id}")
    print(f"검색 기준 문단: {para_numbers}\n")
    
    if not rows:
        print("  (링크된 자료 없음 — 벡터 검색 폴백 필요)")
        return []
    
    for chunk_id, para, section, md, target_start, target_end, link_type in rows:
        target = f"{target_start}~{target_end}" if target_end else target_start
        preview = md[:200].replace('\n', ' ')
        print(f"  [{link_type}] {para or 'N/A'} → 본문 문단 {target}")
        print(f"    {section or ''}")
        print(f"    {preview}...")
        print()
    
    return rows

# Step 3: IE
ie_results = step3_4_find_related("K-IFRS 1115", para_nums, "ie")

# Step 4: BC
bc_results = step3_4_find_related("K-IFRS 1115", para_nums, "bc")


## 적용사례 (IE) — K-IFRS 1115
검색 기준 문단: ['32', '123', 'B55', '125', '34', 'BC405', '38', '121', 'B4', 'B10']

  [body_reference] IE214 → 본문 문단 121
    공시
    IE214		기업이 제공한 용역에 대해 시간당 고정금액을 청구할 수 있기 때문에, 기업은 기업회계기준서 제1115호 문단 B16에 따라 기업이 지금까지 수행을 완료한 부분의 가치에 직접 상응하는 금액을 고객에게 청구할 권리가 있다. 따라서 기업이 기업회계기준서 제1115호 문단 121⑵의 실무적 간편법을 선택할 경우에 공시는 필요하지 않다. **	계약 B*...

  [body_reference] IE326 → 본문 문단 38
    미인도청구약정
    IE326		기계에 대한 통제는 고객이 물리적으로 점유하는 때인 20X9년 12월 31일에 고객에게 이전된다. 기업은 이미 대금을 받았고 고객에게 예비부품에 대한 법적 권리가 있으며 고객이 예비부품을 검사하고 인수한 사실에 주목하면서, 예비부품에 대한 통제가 고객에게 이전된 시점을 판단하기 위해 기업회계기준서 제1115호 문단 38의 지표를 검토한다. 그리...

  [body_reference] IE66 → 본문 문단 38
    기간에 걸쳐 이행하는 수행의무
    IE66		사례 13∼17에서는 기간에 걸쳐 이행하는 수행의무에 관한 기업회계기준서 제1115호 문단 35∼37과 B2∼B13의 요구사항을 설명한다. 그리고 이 사례에서는 다음의 요구사항도 설명한다. 	⑴	기업이 수행하는 대로 고객이 기업의 수행에서 제공되는 효익을 동시에 얻고 소비하는 시점에 관한 기업회계기준서 제1115호 문단 35⑴과 B3∼B4 (사례...

  [body_reference] IE80 → 본문 문단 38
    기간에 걸쳐 이행하는 수행의무
    IE80		기업이 지금까지 수행을 완료한 부분에 대해 지급청구권이 없기 때문에 기업의 수행의무는 기업회

## 정의 + LLM 컨텍스트 조합
기준서의 정의 전체를 가져오고, 4단계 결과를 LLM에 보낼 컨텍스트로 포맷팅.

In [5]:
def build_llm_context(standard_id: str, query: str, 
                       main_chunks, ie_results=None, bc_results=None):
    """4단계 결과를 LLM 컨텍스트로 포맷팅"""
    
    # 정의 가져오기
    row = conn.execute("""
        SELECT title, definitions_text FROM standard_summaries
        WHERE standard_id = %s
    """, (standard_id,)).fetchone()
    title = row[0] if row else standard_id
    definitions = row[1] if row and row[1] else ""
    
    ctx = []
    ctx.append(f"# {standard_id} {title}")
    ctx.append(f"사용자 질문: {query}\n")
    
    # 정의
    if definitions:
        ctx.append("## 용어 정의 [참조]")
        ctx.append(definitions[:3000])  # 최대 3000자
        ctx.append("")
    
    # Step 2: 본문 + AG
    ctx.append("## 적용 문단 [Authoritative, Level 1]")
    for chunk_id, para, comp, section, md, sim in main_chunks:
        label = "본문" if comp == "main" else "적용지침"
        ctx.append(f"\n**문단 {para or 'N/A'}** ({label}, {section or '-'})")
        ctx.append(md[:500])
    ctx.append("")
    
    # Step 3: IE
    if ie_results:
        ctx.append("## 적용사례 [Non-authoritative, Level 4]")
        for chunk_id, para, section, md, ts, te, lt in ie_results:
            ctx.append(f"\n**{para or 'IE'}** ({section or '-'})")
            ctx.append(md[:500])
        ctx.append("")
    
    # Step 4: BC
    if bc_results:
        ctx.append("## 결론도출근거 [Non-authoritative, Level 4]")
        ctx.append("*주의: 결론도출근거는 기준서의 일부를 구성하지 않습니다. 본문과 충돌 시 본문이 우선합니다.*\n")
        for chunk_id, para, section, md, ts, te, lt in bc_results:
            ctx.append(f"\n**{para or 'BC'}** ({section or '-'})")
            ctx.append(md[:500])
    
    full_context = "\n".join(ctx)
    print(f"컨텍스트 길이: {len(full_context):,}자 ({len(full_context)//2:,}토큰 추정)")
    print("=" * 70)
    print(full_context[:2000])
    print("...(truncated)")
    return full_context

# 전체 파이프라인 테스트
context = build_llm_context("K-IFRS 1115", "수행의무 판단 기준", 
                             chunks, ie_results, bc_results)

컨텍스트 길이: 7,047자 (3,523토큰 추정)
# K-IFRS 1115 고객과의 계약에서 생기는 수익
사용자 질문: 수행의무 판단 기준

## 적용 문단 [Authoritative, Level 1]

**문단 32** (본문, 인식)
32		문단 22~30에 따라 식별한 각 수행의무를 (문단 35~37에 따라) 기간에 걸쳐 이행하는지 또는 (문단 38에 따라) 한 시점에 이행하는지를 계약 개시시점에 판단한다. 수행의무가 기간에 걸쳐 이행되지 않는다면, 그 수행의무는 한 시점에 이행되는 것이다.

**문단 123** (본문, 공시)
123		이 기준서를 적용하면서 내린, 고객과의 계약에서 생기는 수익의 금액과 시기의 결정에 유의적인 영향을 미친 판단과 그 판단의 변경을 공시한다. 특히 다음 모두를 결정할 때 내린 판단과 그 판단의 변경을 설명한다.
	⑴	수행의무의 이행 시기(문단 124~125 참조)
	⑵	거래가격과 수행의무에 배분하는 금액(문단 126 참조)
**수행의무를 이행하는 시기를 판단함	**

**문단 B55** (본문, 공시)
B55		라이선스가 구별되지 않는다면, 수행의무(약속한 라이선스 포함)가 기간에 걸쳐 이행되는 수행의무인지 한 시점에 이행되는 수행의무인지를 판단하기 위해 문단 31~38을 적용한다.

**문단 125** (본문, 공시)
125		한 시점에 이행하는 수행의무에 대해, 약속한 재화나 용역을 고객이 언제 통제하는지를 검토하면서 내린 유의적인 판단을 공시한다.
**거래가격과 수행의무에 배분하는 금액을 산정함**

**문단 34** (본문, 인식)
34		고객이 자산을 통제하는지를 판단할 때, 그 자산을 재매입하는 약정을 고려한다(문단 B64~B76 참조).
**기간에 걸쳐 이행하는 수행의무**

**문단 38** (본문, 인식)
38		수행의무가 문단 35~37에 따라 기간에 걸쳐 이행되지 않는다면, 그 수행의무는 한 시점에 이행되는 것이다. 고객이 약속된 자산을 통제하고 기업이 수행의무를 이행하는 시점을 판단하기 위해, 문단

## 추가 테스트 쿼리

In [6]:
def full_pipeline(query: str, include_ie=True, include_bc=True):
    """4-Step 전체 파이프라인 실행"""
    print(f"{'='*70}")
    print(f"QUERY: {query}")
    print(f"{'='*70}\n")
    
    # Step 1
    standards = step1_identify_standard(query)
    selected = standards[0][0]  # top-1
    print(f"\n→ 선택된 기준서: {selected}\n")
    
    # Step 2
    main_chunks, para_nums = step2_search_authoritative(query, selected)
    
    # Step 3 (IE)
    ie = step3_4_find_related(selected, para_nums, "ie") if include_ie else None
    
    # Step 4 (BC)
    bc = step3_4_find_related(selected, para_nums, "bc") if include_bc else None
    
    # 컨텍스트 조합
    print(f"\n{'='*70}")
    print("LLM 컨텍스트:")
    print(f"{'='*70}")
    context = build_llm_context(selected, query, main_chunks, ie, bc)
    
    return context

In [7]:
# 테스트 1: 충당부채 인식 조건 → K-IFRS 1037, 문단 14 기대
ctx = full_pipeline("충당부채 인식 조건")

QUERY: 충당부채 인식 조건

Query: "충당부채 인식 조건"

순위   유사도      기준서                  제목
----------------------------------------------------------------------
1    0.4653   K-IFRS 1037          충당부채 우발부채 우발자산
2    0.3705   K-IFRS 1032          금융상품 표시
3    0.3655   K-IFRS 1019          종업원급여
4    0.3652   K-IFRS 1105          매각예정비유동자산과 중단영업
5    0.3647   K-IFRS 1107          금융상품 공시

→ 선택된 기준서: K-IFRS 1037

Query: "충당부채 인식 조건" in K-IFRS 1037


### 본문 (main)
  [0.414] 문단 N/A (목적)
         이 기준서의 목적은 충당부채, 우발부채, 우발자산을 회계처리하기 위하여 적절한 인식기준과 측정기준을 마련하고, 재무제표이용자가 충당부채 등의 특성, 발생 시기, 금액을 파악할 수 있도록 충분한 정보를 재무제표 주석에 공시하도록 하는 것이다....
  [0.384] 문단 7 (적용범위)
         7	이 기준서에서는 충당부채를 지출하는 시기 또는 금액이 불확실한 부채로 정의하고 있다. 일부 국가에서는 충당부채라는 용어를 감가상각, 자산손상, 대손 등의 항목과 관련하여 사용하고 있다. 이런 항목은 자산 장부금액의 조정에 해당하며, 이 기준서에서는 다루지 아니한다....
  [0.377] 문단 9 (적용범위)
         9	이 기준서는 구조조정(중단영업 포함)과 관련된 충당부채에 적용한다. 특정 구조조정이 중단영업의 정의를 충족하는 경우에는 기업회계기준서 제1105호 ‘매각예정비유동자산과 중단영업’에 따라 추가 공시가 필요할 수 있다....
  [0.373] 문단 1 (적용범위)
         **1	이

In [8]:
# 테스트 2: 리스 식별 → K-IFRS 1116, 문단 9 기대
ctx = full_pipeline("리스 식별")

QUERY: 리스 식별

Query: "리스 식별"

순위   유사도      기준서                  제목
----------------------------------------------------------------------
1    0.2991   K-IFRS 1116          리스
2    0.2435   K-IFRS 1115          고객과의 계약에서 생기는 수익
3    0.2423   K-IFRS 1117          보험계약
4    0.2268   K-IFRS 1041          농림어업
5    0.2221   K-IFRS 1038          무형자산

→ 선택된 기준서: K-IFRS 1116

Query: "리스 식별" in K-IFRS 1116


### 본문 (main)
  [0.363] 문단 N/A (적용사례)
         실무적용지침 | 적용사례·실무적용지침  목차 |  | | --- | --- | | 기업회계기준서 제1116호 '리스'의 적용사례 |  | | 리스를 식별함 | 문단번호 | | 사례 1: 철도차량 | IE2 | | 사례 2: 영업 허가 공간 | IE3 | | 사례 3: ...
  [0.331] 문단 106 (판매후리스 거래)
         106	그러나 이자율지표 개혁에서 요구된 리스변경에 추가하여 리스변경이 이루어진 경우, 리스이용자는 동시에 이루어진 모든 리스변경(이자율지표 개혁에서 요구하는 것을 포함)을 회계처리하기 위해 적용가능한 이 기준서의 요구사항을 적용한다. **부록 A. 용어의 정의** |...
  [0.328] 문단 B8 (판매후리스 거래)
         B8		소액 기초자산의 예로는 태블릿·개인 컴퓨터, 소형 사무용 가구, 전화기를 들 수 있다. **리스를 식별함(문단 9~11) **...
  [0.325] 문단 9 (리스를 식별함(문단 B9~B33))
         **9		계약의 약정시점에, 계약 자체가 리스인지, 계약이 리스를 포함하는지를

In [9]:
# 테스트 3: 금융자산 분류 → K-IFRS 1109, 문단 4.1.1~4.1.5 기대
ctx = full_pipeline("금융자산 분류")

QUERY: 금융자산 분류

Query: "금융자산 분류"

순위   유사도      기준서                  제목
----------------------------------------------------------------------
1    0.4101   K-IFRS 1105          매각예정비유동자산과 중단영업
2    0.4059   K-IFRS 1032          금융상품 표시
3    0.3859   K-IFRS 1036          자산손상
4    0.3849   K-IFRS 1107          금융상품 공시
5    0.3391   K-IFRS 1002          재고자산

→ 선택된 기준서: K-IFRS 1105

Query: "금융자산 분류" in K-IFRS 1105


### 본문 (main)
  [0.389] 문단 45 (기준서 등의 대체)
         45	[한국회계기준원 회계기준위원회가 삭제함] **부록 A** **용어의 정의** | 가능성이 높은 | 발생하지 않을 가능성보다 발생할 가능성이 높은 | | --- | --- | | 가능성이 매우 높은 | 발생하지 않을 가능성보다 발생할 가능성이 유의적으로 더 높은 |...
  [0.388] 문단 40 (표시와 공시)
         40	과거 재무상태표에 매각예정으로 분류된 비유동자산 또는 처분자산집단에 포함된 자산과 부채의 금액은 최근 재무상태표의 분류를 반영하기 위하여 재분류하거나 재작성하지 아니한다. **추가 공시**...
  [0.388] 문단 3 (적용범위)
         3	기업회계기준서 제1001호 ‘재무제표 표시’에 따라 비유동자산으로 분류하는 자산은 이 기준서의 매각예정분류기준을 충족할 때까지는 ***유동자산***으로 재분류할 수 없다. 통상적으로 비유동자산으로 분류하는 자산을 매각만을 목적으로 취득한 경우라 하더라도 이 기준서의...
  [0.386] 문단 18 (적용범위)
         18	자산(또는 처분자산

In [10]:
# 연결 종료
conn.close()
print("DB 연결 종료")

DB 연결 종료
